# METR-LA traffic-speed forecasting with CLTFP

This self-contained Colab notebook implements **CLTFP** by Wu & Tan (arXiv:1612.01022), evaluated on METR-LA. Its CNN convolves across the sensor/location dimension, not time. It includes spatial CNN, short-term LSTM, connected weekly-to-daily periodic LSTM, feature-level fusion, and L1-regularized regression. CLTFP is not a graph model and never uses adjacency data.

## 1. Colab imports, reproducibility, and configuration

Set `DATA_DIR` to the directory containing Kaggle's `METR-LA.h5`. The batch size, epoch limit, and early stopping are implementation settings for this METR-LA experiment.

In [ ]:
from pathlib import Path
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils import data

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Change this one visible setting for /content or mounted Google Drive.
DATA_DIR = Path("datasets/metr-la")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_TRAINING = True
NUM_SENSORS = 207
HORIZON_STEPS = (3, 6, 12)
CLTFP_HISTORY_STEPS = 15
DAILY_OFFSET_STEPS = 288
WEEKLY_OFFSET_STEPS = 7 * DAILY_OFFSET_STEPS
PERIODIC_RADIUS = 6
CLTFP_BATCH_SIZE = 64
MAX_EPOCHS = 50
CLTFP_PATIENCE = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
print(f"Selected device: {DEVICE}")
print(f"DATA_DIR: {DATA_DIR}")
print(
    "CLTFP implementation settings: batch_size=64, max_epochs=50, patience=5; Adamax default LR is used."
)

## 2. METR-LA loading, causal missing-value handling, chronological split, and training-only normalization

In [ ]:
def read_metr_la_hdf(path: Path):
    """Read standard METR-LA HDF5, including legacy pandas frequency metadata."""
    try:
        return pd.read_hdf(path)
    except TypeError as exc:
        # Some pandas/PyTables combinations reject legacy numpy.bytes_ frequency metadata.
        try:
            import tables
        except ImportError as import_exc:
            raise RuntimeError(
                "Could not read METR-LA.h5; install the standard Colab PyTables dependency."
            ) from import_exc
        try:
            with tables.open_file(path) as handle:
                values = handle.get_node("/df/block0_values").read()
                columns = handle.get_node("/df/block0_items").read()
                timestamps_ns = handle.get_node("/df/axis1").read()
            columns = [
                column.decode() if isinstance(column, bytes) else str(column)
                for column in columns
            ]
            return pd.DataFrame(
                values, index=pd.to_datetime(timestamps_ns), columns=columns
            )
        except Exception as fallback_exc:
            raise RuntimeError(
                f"Unable to read METR-LA HDF5 at {path}: {fallback_exc}"
            ) from exc


data_path = DATA_DIR / "METR-LA.h5"
if not data_path.is_file():
    raise FileNotFoundError(
        f"METR-LA.h5 is required at {data_path}. Set DATA_DIR to the Kaggle dataset directory."
    )

speed_frame = read_metr_la_hdf(data_path)
if not isinstance(speed_frame.index, pd.DatetimeIndex):
    speed_frame.index = pd.DatetimeIndex(speed_frame.index)
speed_frame = speed_frame.sort_index().astype(np.float32)
if speed_frame.shape[1] != NUM_SENSORS:
    raise ValueError(
        f"Expected {NUM_SENSORS} METR-LA sensors, found {speed_frame.shape[1]}."
    )
print("METR-LA validation for CLTFP")
print(f"  data shape: {speed_frame.shape}")
print(f"  timestamp range: {speed_frame.index.min()} to {speed_frame.index.max()}")
print(f"  sensors: {speed_frame.shape[1]}")
print("  adjacency used by CLTFP: NO")

raw_speed = speed_frame.to_numpy(dtype=np.float32).copy()
observed_mask = np.isfinite(raw_speed) & (raw_speed > 0.0)
raw_speed[~observed_mask] = np.nan
n_total = len(raw_speed)
train_end, val_end = int(0.70 * n_total), int(0.80 * n_total)
if not (0 < train_end < val_end < n_total):
    raise ValueError("Chronological 70/10/20 split could not be constructed.")


def causal_forward_fill(values):
    """Fill only from past values; leading unavailable readings stay NaN."""
    filled = values.copy()
    for t in range(1, len(filled)):
        missing = ~np.isfinite(filled[t])
        filled[t, missing] = filled[t - 1, missing]
    return filled


causal_speed = causal_forward_fill(raw_speed)
train_observed = observed_mask[:train_end]
train_values = causal_speed[:train_end]
train_mean = float(np.nanmean(train_values[train_observed]))
train_std = float(np.nanstd(train_values[train_observed]))
if not np.isfinite(train_mean) or not np.isfinite(train_std) or train_std <= 0:
    raise ValueError("Training-only Z-score statistics are invalid.")
normalized_speed = (causal_speed - train_mean) / train_std
print(
    f"Chronological split: train=[0,{train_end}), val=[{train_end},{val_end}), test=[{val_end},{n_total})"
)
print(f"Training-only Z-score mean={train_mean:.4f}, std={train_std:.4f}")

## 3. Leakage-safe CLTFP windows and original-unit metrics

In [ ]:
class ForecastDataset(data.Dataset):
    def __init__(
        self, normalized, observed, target_range, history, horizon, cltfp=False
    ):
        self.normalized = normalized
        self.observed = observed
        self.history = history
        self.horizon = horizon
        self.cltfp = cltfp
        candidates = []
        for target in range(target_range.start, target_range.stop):
            origin = target - horizon
            recent = slice(origin - history + 1, origin + 1)
            if recent.start < 0 or not np.isfinite(normalized[recent]).all():
                continue
            if cltfp:
                daily = slice(
                    target - DAILY_OFFSET_STEPS - PERIODIC_RADIUS,
                    target - DAILY_OFFSET_STEPS + PERIODIC_RADIUS + 1,
                )
                weekly = slice(
                    target - WEEKLY_OFFSET_STEPS - PERIODIC_RADIUS,
                    target - WEEKLY_OFFSET_STEPS + PERIODIC_RADIUS + 1,
                )
                if daily.start < 0 or weekly.start < 0:
                    continue
                if (
                    not np.isfinite(normalized[daily]).all()
                    or not np.isfinite(normalized[weekly]).all()
                ):
                    continue
            candidates.append(target)
        if not candidates:
            raise ValueError(
                "No valid windows were created; check timestamps and causal missing-value availability."
            )
        self.targets = np.asarray(candidates, dtype=np.int64)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        target = int(self.targets[index])
        origin = target - self.horizon
        recent = self.normalized[origin - self.history + 1 : origin + 1]
        y = np.nan_to_num(self.normalized[target], nan=0.0).astype(np.float32)
        mask = self.observed[target].astype(np.float32)
        item = {
            "recent": torch.from_numpy(recent.astype(np.float32)),
            "target": torch.from_numpy(y),
            "mask": torch.from_numpy(mask),
        }
        if self.cltfp:
            daily = self.normalized[
                target - DAILY_OFFSET_STEPS - PERIODIC_RADIUS : target
                - DAILY_OFFSET_STEPS
                + PERIODIC_RADIUS
                + 1
            ]
            weekly = self.normalized[
                target - WEEKLY_OFFSET_STEPS - PERIODIC_RADIUS : target
                - WEEKLY_OFFSET_STEPS
                + PERIODIC_RADIUS
                + 1
            ]
            item["daily"] = torch.from_numpy(daily.astype(np.float32))
            item["weekly"] = torch.from_numpy(weekly.astype(np.float32))
        return item


def make_loader(split, history, horizon, cltfp, batch_size, shuffle):
    ranges = {
        "train": range(0, train_end),
        "val": range(train_end, val_end),
        "test": range(val_end, n_total),
    }
    ds = ForecastDataset(
        normalized_speed, observed_mask, ranges[split], history, horizon, cltfp
    )
    return data.DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=PIN_MEMORY
    ), ds


def masked_mse(prediction, target, mask):
    return ((prediction - target).square() * mask).sum() / mask.sum().clamp_min(1.0)


def original_unit_metrics(prediction, target, mask):
    prediction = prediction * train_std + train_mean
    target = target * train_std + train_mean
    valid = (
        mask.astype(bool) & np.isfinite(prediction) & np.isfinite(target) & (target > 0)
    )
    if not valid.any():
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan}
    error = prediction[valid] - target[valid]
    return {
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error**2))),
        "MAPE": float(np.mean(np.abs(error) / target[valid]) * 100.0),
    }

## 4. CLTFP model definition and identity verification

In [ ]:
class SReLU(nn.Module):
    """Trainable S-shaped ReLU with channel-wise thresholds and slopes."""

    def __init__(self, channels):
        super().__init__()
        self.t_left = nn.Parameter(torch.full((1, channels, 1), -1.0))
        self.t_right = nn.Parameter(torch.full((1, channels, 1), 1.0))
        self.a_left = nn.Parameter(torch.full((1, channels, 1), 0.2))
        self.a_right = nn.Parameter(torch.full((1, channels, 1), 0.2))

    def forward(self, x):
        left = self.t_left + self.a_left * (x - self.t_left)
        right = self.t_right + self.a_right * (x - self.t_right)
        return torch.where(
            x <= self.t_left, left, torch.where(x >= self.t_right, right, x)
        )


class CLTFP(nn.Module):
    def __init__(self, sensors=NUM_SENSORS):
        super().__init__()
        self.sensors = sensors
        self.conv1 = nn.Conv1d(15, 30, kernel_size=3)
        self.conv2 = nn.Conv1d(30, 30, kernel_size=3)
        self.conv3 = nn.Conv1d(30, 30, kernel_size=2)
        self.srelu1, self.srelu2, self.srelu3 = SReLU(30), SReLU(30), SReLU(30)
        self.short_lstm = nn.LSTM(input_size=sensors, hidden_size=40, batch_first=True)
        self.periodic_lstm = nn.LSTM(
            input_size=sensors, hidden_size=25, batch_first=True
        )
        spatial_size = 30 * (sensors - 3 - 3 - 2 + 3)
        fusion_size = spatial_size + 15 * 40 + 13 * 25 + 13 * 25
        self.regression = nn.Linear(fusion_size, sensors)

    def forward(self, recent, weekly, daily):
        spatial = self.srelu3(
            self.conv3(self.srelu2(self.conv2(self.srelu1(self.conv1(recent)))))
        ).flatten(1)
        short_features, _ = self.short_lstm(recent)
        weekly_features, weekly_state = self.periodic_lstm(weekly)
        daily_features, _ = self.periodic_lstm(daily, weekly_state)
        fused = torch.cat(
            (
                spatial,
                short_features.flatten(1),
                weekly_features.flatten(1),
                daily_features.flatten(1),
            ),
            dim=1,
        )
        return self.regression(fused)

    def regression_l1(self):
        return self.regression.weight.abs().sum()


def verify_cltfp():
    model = CLTFP()
    assert len([model.conv1, model.conv2, model.conv3]) == 3
    assert [
        (model.conv1.in_channels, model.conv1.out_channels, model.conv1.kernel_size[0]),
        (model.conv2.in_channels, model.conv2.out_channels, model.conv2.kernel_size[0]),
        (model.conv3.in_channels, model.conv3.out_channels, model.conv3.kernel_size[0]),
    ] == [(15, 30, 3), (30, 30, 3), (30, 30, 2)]
    print(
        "model name: CLTFP\nrecent history: 15\nCNN layers: 3\nCNN filters: 30 / 30 / 30\nkernels: 3 / 3 / 2\nactivation: SReLU\npooling: none\nshort LSTM hidden size: 40\nperiodic LSTM hidden size: 25\ndaily radius: 6\nweekly radius: 6\nregression L1: 0.002\noptimizer: Adamax\nadjacency used by model: NO"
    )
    return model


_ = verify_cltfp()

## 5. CLTFP training and held-out evaluation

In [ ]:
def move_batch(batch):
    return {
        key: value.to(DEVICE, non_blocking=PIN_MEMORY) for key, value in batch.items()
    }


def evaluate_model(model, loader, model_name):
    model.eval()
    losses = []
    predictions = []
    targets = []
    masks = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch)
            if model_name == "CLTFP":
                pred = model(batch["recent"], batch["weekly"], batch["daily"])
            else:
                pred = model(batch["recent"])
            loss = masked_mse(pred, batch["target"], batch["mask"])
            if not torch.isfinite(loss):
                raise FloatingPointError(f"{model_name} produced a non-finite loss.")
            losses.append(loss.item())
            predictions.append(pred.cpu().numpy())
            targets.append(batch["target"].cpu().numpy())
            masks.append(batch["mask"].cpu().numpy())
    return (
        float(np.mean(losses)),
        np.concatenate(predictions),
        np.concatenate(targets),
        np.concatenate(masks),
    )


def train_cltfp(horizon):
    train_loader, _ = make_loader(
        "train", CLTFP_HISTORY_STEPS, horizon, True, CLTFP_BATCH_SIZE, True
    )
    val_loader, _ = make_loader(
        "val", CLTFP_HISTORY_STEPS, horizon, True, CLTFP_BATCH_SIZE, False
    )
    model = CLTFP().to(DEVICE)
    optimizer = torch.optim.Adamax(
        model.parameters()
    )  # PyTorch default LR; not claimed as a paper value.
    best_val, wait, train_curve, val_curve = math.inf, 0, [], []
    checkpoint = RESULTS_DIR / f"cltfp_h{horizon}_best.pth"
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_losses = []
        for batch in train_loader:
            batch = move_batch(batch)
            optimizer.zero_grad(set_to_none=True)
            pred = model(batch["recent"], batch["weekly"], batch["daily"])
            loss = (
                masked_mse(pred, batch["target"], batch["mask"])
                + 0.002 * model.regression_l1()
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("CLTFP produced a non-finite training loss.")
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        val_loss, *_ = evaluate_model(model, val_loader, "CLTFP")
        train_curve.append(float(np.mean(epoch_losses)))
        val_curve.append(val_loss)
        print(
            f"CLTFP h={horizon}, epoch={epoch:02d}, train={train_curve[-1]:.5f}, val={val_loss:.5f}"
        )
        if val_loss < best_val:
            best_val, wait = val_loss, 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "horizon": horizon,
                    "validation_loss": val_loss,
                    "mean": train_mean,
                    "std": train_std,
                },
                checkpoint,
            )
        else:
            wait += 1
            if wait >= CLTFP_PATIENCE:
                print(f"CLTFP h={horizon}: early stopping after {epoch} epochs.")
                break
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE)["model"])
    return model, train_curve, val_curve


cltfp_runs = {}
if RUN_TRAINING:
    for horizon in HORIZON_STEPS:
        cltfp_runs[horizon] = train_cltfp(horizon)

## 6. CLTFP test metrics and visualizations

In [ ]:
if RUN_TRAINING:
    records = []
    representative = None
    for horizon, (model, train_curve, val_curve) in cltfp_runs.items():
        test_loader, _ = make_loader(
            "test", CLTFP_HISTORY_STEPS, horizon, True, CLTFP_BATCH_SIZE, False
        )
        _, pred, target, mask = evaluate_model(model, test_loader, "CLTFP")
        metric = original_unit_metrics(pred, target, mask)
        records.append({"Model": "CLTFP", "Horizon": f"{horizon * 5} min", **metric})
        if representative is None:
            representative = (pred, target, mask, horizon)
        plt.figure(figsize=(7, 3))
        plt.plot(train_curve, label="train")
        plt.plot(val_curve, label="validation")
        plt.title(f"CLTFP training/validation loss — {horizon * 5} min")
        plt.xlabel("Epoch")
        plt.ylabel("masked MSE")
        plt.legend()
        plt.show()
    comparison = pd.DataFrame(
        records, columns=["Model", "Horizon", "MAE", "RMSE", "MAPE"]
    )
    display(comparison)
    comparison.to_csv(RESULTS_DIR / "cltfp_model_comparison.csv", index=False)
    pred, target, mask, horizon = representative
    sensor = int(np.flatnonzero(mask[0])[0])
    plt.figure(figsize=(10, 4))
    plt.plot(target[:200, sensor] * train_std + train_mean, label="actual")
    plt.plot(pred[:200, sensor] * train_std + train_mean, label="predicted")
    plt.title(f"CLTFP held-out prediction — {horizon * 5} min, sensor {sensor}")
    plt.xlabel("Test sample")
    plt.ylabel("Speed")
    plt.legend()
    plt.show()
    print(f"Saved {RESULTS_DIR / 'cltfp_model_comparison.csv'}")
else:
    print("RUN_TRAINING=False: set it to True to train CLTFP at all three horizons.")